# Transformer Baselines

This notebook runs the final Hugging Face BERT-family comparison across multiple random seeds and creates analysis tables:

- `total_results`: all evaluated transformer runs across all seeds.
- `per_seed_report_results`: the best validation result per seed, model, and variant.
- `report_results`: mean/std validation metrics across seeds.
- `epoch_history`: per-epoch validation metrics for training curves.

The task predicts sentiment labels `0..4`, with validation score `1 - MAE / 4`.


In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys

import pandas as pd
import torch

EXPERIMENT_KIND = "TRANSFORMER_BASELINES"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = Path("experiments/transformers") / f"{RUN_ID}_{EXPERIMENT_KIND}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = Path("data/train.csv")
VALIDATION_SIZE = 0.1
SEEDS = [42, 43, 44]
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
LR_SCHEDULER_TYPE = "linear"
LOGGING_STEPS = 500
USE_FP16 = True

EXPERIMENT_DIR, SEEDS


In [ ]:
models = [
    {"model_name": "distilbert-base-uncased", "variant": "fine_tuned"},
    {"model_name": "bert-base-uncased", "variant": "fine_tuned"},
    {"model_name": "roberta-base", "variant": "fine_tuned"},
    {"model_name": "xlm-roberta-large", "variant": "fine_tuned", "batch_size": 8, "eval_batch_size": 16},
    {"model_name": "nlptown/bert-base-multilingual-uncased-sentiment", "variant": "fine_tuned"},
    {"model_name": "nlptown/bert-base-multilingual-uncased-sentiment", "variant": "eval_only"},
]

models


## Run Experiments


In [ ]:
completed_runs = []

for spec in models:
    model_name = spec["model_name"]
    variant = spec["variant"]
    run_name = f"{model_name.replace('/', '__')}__{variant}"
    train_batch_size = spec.get("batch_size", BATCH_SIZE)
    eval_batch_size = spec.get("eval_batch_size", EVAL_BATCH_SIZE)

    for seed in SEEDS:
        run_dir = EXPERIMENT_DIR / run_name / f"seed_{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            sys.executable,
            "-m",
            "baselines.train_review_model",
            "--model-name",
            model_name,
            "--experiment-name",
            f"transformer__{run_name}",
            "--train-path",
            str(TRAIN_PATH),
            "--output-dir",
            str(run_dir),
            "--validation-size",
            str(VALIDATION_SIZE),
            "--random-state",
            str(seed),
            "--max-length",
            str(MAX_LENGTH),
            "--epochs",
            str(EPOCHS),
            "--batch-size",
            str(train_batch_size),
            "--eval-batch-size",
            str(eval_batch_size),
            "--learning-rate",
            str(LEARNING_RATE),
            "--weight-decay",
            str(WEIGHT_DECAY),
            "--warmup-ratio",
            str(WARMUP_RATIO),
            "--lr-scheduler-type",
            LR_SCHEDULER_TYPE,
            "--logging-steps",
            str(LOGGING_STEPS),
        ]
        if variant == "eval_only":
            cmd.append("--eval-only")
        if USE_FP16:
            cmd.append("--fp16")

        print(" ".join(cmd))
        subprocess.run(cmd, check=True)
        completed_runs.append({**spec, "seed": seed, "run_name": run_name, "run_dir": run_dir})

completed_runs


## Total Analysis


In [ ]:
result_frames = []
for run in completed_runs:
    frame = pd.read_csv(run["run_dir"] / "transformer_results.csv")
    frame.insert(0, "seed", run["seed"])
    result_frames.append(frame)

results = pd.concat(result_frames, ignore_index=True)
results.to_csv(EXPERIMENT_DIR / "transformer_results.csv", index=False)

total_results = results.sort_values(
    ["status", "cil_score"], ascending=[False, False]
).reset_index(drop=True)
total_results.to_csv(EXPERIMENT_DIR / "total_analysis.csv", index=False)
total_results


## Report Analysis


In [ ]:
ok_results = results[results["status"] == "ok"].copy()
per_seed_report_results = (
    ok_results.sort_values("cil_score", ascending=False)
    .groupby(["seed", "model", "variant"], as_index=False)
    .first()
    .sort_values(["model", "variant", "seed"])
    .reset_index(drop=True)
)
per_seed_report_results.to_csv(EXPERIMENT_DIR / "per_seed_report_analysis.csv", index=False)

report_results = (
    per_seed_report_results
    .groupby(["model", "variant"])[["cil_score", "mae", "accuracy", "macro_f1"]]
    .agg(["mean", "std"])
    .reset_index()
)
report_results.columns = [
    column[0] if column[1] == "" else f"{column[0]}_{column[1]}"
    for column in report_results.columns.to_flat_index()
]
report_results = report_results.sort_values("cil_score_mean", ascending=False).reset_index(drop=True)
report_results.to_csv(EXPERIMENT_DIR / "report_analysis.csv", index=False)
report_results


## Epoch History


In [ ]:
history_frames = []
for run in completed_runs:
    frame = pd.read_csv(run["run_dir"] / "transformer_epoch_history.csv")
    frame.insert(0, "seed", run["seed"])
    history_frames.append(frame)

epoch_history = pd.concat(history_frames, ignore_index=True)
epoch_history = epoch_history.sort_values(["experiment", "seed", "epoch"]).reset_index(drop=True)
epoch_history.to_csv(EXPERIMENT_DIR / "transformer_epoch_history.csv", index=False)
epoch_history.to_csv(EXPERIMENT_DIR / "epoch_analysis.csv", index=False)
epoch_history


## Analysis Artifacts


In [ ]:
analysis_dir = EXPERIMENT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

transformer_bar_plot = analysis_dir / "transformer_model_comparison.pdf"
subprocess.run(
    [
        sys.executable,
        "-m",
        "baselines.analysis.plot_transformer_report_bars",
        "--input",
        str(EXPERIMENT_DIR / "per_seed_report_analysis.csv"),
        "--output-dir",
        str(analysis_dir),
        "--prefix",
        "transformer_model_comparison",
        "--metric",
        "mae",
        "--caption",
        "Transformer baseline comparison across three seeds.",
        "--label",
        "fig:transformer-baselines",
    ],
    check=True,
)

transformer_bar_plot
